In [0]:
#import required module 

from pyspark.sql.functions import *
from pyspark.sql import *
from pyspark.sql.types import *

In [0]:
#Define Storage_name
Storage_name = "stytdev001"

spark.conf.set(
    f"fs.azure.account.auth.type.{Storage_name}.dfs.core.windows.net",
    "OAuth"
)

#configure Oauth authentication

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{Storage_name}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{Storage_name}.dfs.core.windows.net",dbutils.secrets.get(scope="app_yt_scope", key="app-yt-client-id"))


spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{Storage_name}.dfs.core.windows.net",dbutils.secrets.get(scope="app_yt_scope", key="app-yt-client-secrets"))


spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{Storage_name}.dfs.core.windows.net",
    "https://login.microsoftonline.com/65583477-d432-469c-b4af-ab046853916f/oauth2/v2.0/token",
)


In [0]:
#define widgets for table_name and storage path

dbutils.widgets.text('TABLE_NAME','')
dbutils.widgets.text('STORAGE_PATH','')

TABLE_NAME = dbutils.widgets.get('TABLE_NAME')
STORAGE_PATH = dbutils.widgets.get('STORAGE_PATH')

In [0]:
#define source and sink path and write data into gold layer 

source_path = f"abfss://global@{Storage_name}.dfs.core.windows.net/silver/"
sink_path = f"abfss://global@{Storage_name}.dfs.core.windows.net/gold/"


# STORAGE_FOLDER_NAME = TABLE_NAME + '_output/'

full_source_path = source_path + STORAGE_PATH + '/' + TABLE_NAME + '/'
full_sink_path = sink_path + STORAGE_PATH + '/' + "dim_" + TABLE_NAME

print(full_source_path)
print(full_sink_path)

df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(full_source_path)

df.write.format("csv").mode("overwrite").save(full_sink_path)

